## DataFrame Functional Basics

- DataFrame operations are automatically distributed across partitions for parallel processing (speed).  This is why when dataframes are written to disk as csv, parquet, etc, they are written in 'fileparts', unless coalesce into a single file.  Each partition 'writes its part' of the final product.

- Methods return a new dataframe as dataframe are immutable.

- Each operation builds upon a logical plan until an 'action' (count, write, etc), triggers an execution

## DataFrame Transformation Methods


```
- select()
- filter(), where()
- groupBy()
- orderBy(), sort()
- join()
```

## DataFrame Missing Values

![](/Volumes/workspace/pyspark_learning/raw_files/images/missing_values.png)

## How to Reference Data (which approach do I use?)

![](/Volumes/workspace/pyspark_learning/raw_files/images/reference_data.png)

using 'col' is the most flexible, but where I can I use attribute due to ease of typing it all out

**When you create a schema with python StructTypes they are dot-notation (attribute) accessible**

column object methods are methods on the column object.  These are different than functions which take columns as arguments.
Example:


method
```
col("name").cointains("manager")
```

function
```
coalesce(col("name"), col("nickname"))
```

## GroupBy in DataFrames

- groupBy returns a grouped object which then allows for (and requires) some sort of aggregation method to be applied.

```
df.groupBy("department").count()
```

Aggregation methods:

```
- count()
- sum(col)
- avg(col)
- min(col)/max(col)
```

Multiple aggregation methods can be applied to a grouped object via the use of the ```agg()``` function

```
df.groupBy("department").agg(sum("salary"), avg("age")).display()
```

Alternate dictionary syntax
```
df.groupBy("department").agg({
  "salary": "sum",
  "age": "avg"
}).display()
```


## Relational Operations

- **UNION** - Includes all unique elements from both sets (no dupes)
- **UNIONBYNAME** - Combines dataframes by matching column names
- **INTERSECTION** - Includes only the elements in each set
- **SUBTRACTION** - Includes only the difference from left to right where returns only what is unqiue to left

## full outer join

In [0]:
# A full outer join returns all records from both DataFrames, matching rows where possible.
# If there is no match, the missing side will contain nulls.

df1 = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")
], ["id", "name"])

df2 = spark.createDataFrame([
    (2, "Sales"),
    (3, "Marketing"),
    (4, "Finance")
], ["id", "department"])

# Perform a full outer join on 'id'
result = df1.join(df2, on="id", how="outer")
display(result)

## Left_anti Join vs Subtract

In [0]:
# A left_anti join returns only the rows from the left DataFrame that do not have a match in the right DataFrame.

left_anti_result = df1.join(df2, on="id", how="left_anti")
display(left_anti_result)

In [0]:
# left_anti join returns rows from df1 where 'id' does not exist in df2.
left_anti_result = df1.join(df2, on="id", how="left_anti")
display(left_anti_result)

# subtract returns rows from df1 that are not present in df2, considering all columns.
subtract_result = df1.subtract(df2)
display(subtract_result)

## Complex Data - Nested Data

Nested JSON is an example of complex data

![](/Volumes/workspace/pyspark_learning/raw_files/images/complex_data.png)

Below is an example of how to parse/apply schema to nested json.

Notice the MapType() takes a type for both key and value

![](/Volumes/workspace/pyspark_learning/raw_files/images/json.png)


```col("struct_column.field_name')``` or ``` getField()``` allows for direct access to nested data

**explode**

The ```explode()``` function unnests data contained within an array

- explode on large data sets is dangerous as it may overwhelm memory when one row becomes 'n' rows

In [0]:

'''
below is an explode example were the array is broken out into individual records
'''
from pyspark.sql.functions import explode

data = [
    (1, ["a", "b", "c"]),
    (2, ["x", "y"])
]

schema = "id integer, items array<string>"

df1 = spark.createDataFrame(data, schema=schema)
df1.display()
df1.select('id', explode("items").alias('item')).display()

**collect_list** and **collect_set**

- ```collect_list()``` builds an array from column values, commonly used by groupBy
can be memory intensive

- ```collect_set()```  works the same as collect_list but builds an array of unique values only
use this when dupes are not needed and the order does not matter.  This is a less memory intensive action because dupes are removed and not held in memory, also less shuffle overhead for the same reason

## Broadcast on joins

In [0]:
# The broadcast hint forces Spark to broadcast the specified DataFrame during a join, optimizing performance for joins with small tables.
# The network shuffle phase is eliminated

from pyspark.sql.functions import broadcast

# Example: broadcasting df2 in a join with df1
broadcast_result = df1.join(broadcast(df2), on="id", how="inner")
display(broadcast_result)